In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
from google.colab import drive

try:
    import holidays
    from prophet import Prophet
    from statsmodels.tsa.arima.model import ARIMA
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Input
except ImportError:
    print("Menginstall library yang diperlukan...")
    !pip install prophet holidays
    import holidays
    from prophet import Prophet
    from statsmodels.tsa.arima.model import ARIMA
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Input

warnings.filterwarnings('ignore')

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Database.xlsx'

sheets = pd.read_excel(file_path, sheet_name=None)
df_raw = pd.concat(sheets.values(), ignore_index=True)

print(f"Kolom ditemukan: {df_raw.columns.tolist()}")

col_tanggal = 'REGISTER'
col_poli = 'POLI'

df = df_raw[[col_tanggal, col_poli]].copy()
df.columns = ['Tanggal', 'Poliklinik']
df['Tanggal'] = pd.to_datetime(df['Tanggal'])

def preprocess_poli(data, nama_poli):
    df_temp = data[data['Poliklinik'] == nama_poli].copy()
    df_daily = df_temp.groupby('Tanggal').size().reset_index(name='y')
    df_daily.columns = ['ds', 'y']

    id_holidays = holidays.Indonesia()
    df_daily = df_daily[df_daily['ds'].dt.dayofweek < 5]
    df_daily = df_daily[~df_daily['ds'].apply(lambda x: x in id_holidays)]

    df_daily = df_daily[df_daily['y'] > 5]
    return df_daily.sort_values('ds')

def calculate_metrics(y_true, y_pred, label, poli):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    return {'Poliklinik': poli, 'Model': label, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE (%)': round(mape, 2)}

daftar_poli = [
    'Poliklinik Penyakit Dalam',
    'Poliklinik Rehabilitasi Medik',
    'Poliklinik Jantung dan Pembuluh Darah',
    'Poliklinik Paru',
    'Poliklinik Umum MCU'
]

summary_results = []

for poli in daftar_poli:
    print(f"\n--- Memproses: {poli} ---")
    df_eval = preprocess_poli(df, poli)

    if len(df_eval) < 20:
        print(f"Skipping {poli} karena data tidak cukup.")
        continue

    split_idx = int(len(df_eval) * 0.8)
    train = df_eval.iloc[:split_idx]
    test = df_eval.iloc[split_idx:]
    y_test = test['y'].values

    m_p = Prophet(weekly_seasonality=True, yearly_seasonality=True)
    m_p.add_country_holidays(country_name='ID')
    m_p.fit(train)
    pred_p = m_p.predict(test[['ds']])['yhat'].values
    summary_results.append(calculate_metrics(y_test, pred_p, 'Prophet', poli))

    history = [x for x in train['y'].values]
    preds_a = []
    for t in range(len(test)):
        model_a = ARIMA(history, order=(7, 1, 1))
        model_a_fit = model_a.fit()
        preds_a.append(model_a_fit.forecast()[0])
        history.append(y_test[t])
    summary_results.append(calculate_metrics(y_test, np.array(preds_a), 'ARIMA', poli))

    scaler = MinMaxScaler()
    scaled_train = scaler.fit_transform(train['y'].values.reshape(-1, 1))

    def create_sequences(data, window=7):
        X, y = [], []
        for i in range(len(data) - window):
            X.append(data[i:i+window])
            y.append(data[i+window])
        return np.array(X), np.array(y)

    X_tr, y_tr = create_sequences(scaled_train)
    X_tr = X_tr.reshape((X_tr.shape[0], X_tr.shape[1], 1))

    model_l = Sequential([Input(shape=(7, 1)), LSTM(50, activation='relu'), Dense(1)])
    model_l.compile(optimizer='adam', loss='mse')
    model_l.fit(X_tr, y_tr, epochs=10, verbose=0)

    inputs = scaler.transform(df_eval['y'].values.reshape(-1, 1))
    inputs_test = inputs[len(inputs) - len(test) - 7:]
    X_ts, _ = create_sequences(inputs_test)
    X_ts = X_ts.reshape((X_ts.shape[0], X_ts.shape[1], 1))
    pred_l = scaler.inverse_transform(model_l.predict(X_ts, verbose=0)).flatten()
    summary_results.append(calculate_metrics(y_test, pred_l, 'LSTM', poli))

df_final = pd.DataFrame(summary_results)
print("\n" + "="*70)
print("   TABEL EVALUASI KOMPARASI 3 MODEL (5 POLIKLINIK)")
print("="*70)
print(df_final.sort_values(by=['Poliklinik', 'MAE']))
print("="*70)

df_final.to_excel('Hasil_Evaluasi_5_Poli.xlsx', index=False)
print("✅ Tabel hasil berhasil disimpan sebagai 'Hasil_Evaluasi_5_Poli.xlsx'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kolom ditemukan: ['NO', 'NO RAWAT', 'REGISTER', 'JAM \nKEDATANGAN', 'JAM PELAYANAN', 'WAKTU TUNGGU', 'DOKTER', 'RM', 'NAMA', 'JK', 'UMUR', 'POLI', 'JAMINAN', 'ALAMAT', 'STATUS', 'NO HP', 'ICD X']

--- Memproses: Poliklinik Penyakit Dalam ---


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.



--- Memproses: Poliklinik Rehabilitasi Medik ---


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.



--- Memproses: Poliklinik Jantung dan Pembuluh Darah ---


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.



--- Memproses: Poliklinik Paru ---


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.



--- Memproses: Poliklinik Umum MCU ---

   TABEL EVALUASI KOMPARASI 3 MODEL (5 POLIKLINIK)
                               Poliklinik    Model    MAE   RMSE  MAPE (%)
7   Poliklinik Jantung dan Pembuluh Darah    ARIMA  11.05  14.00     34.47
8   Poliklinik Jantung dan Pembuluh Darah     LSTM  11.84  15.06     34.05
6   Poliklinik Jantung dan Pembuluh Darah  Prophet  12.27  15.01     42.29
10                        Poliklinik Paru    ARIMA   5.64   7.47     33.58
9                         Poliklinik Paru  Prophet   5.71   7.49     35.02
11                        Poliklinik Paru     LSTM   5.97   7.66     37.17
1               Poliklinik Penyakit Dalam    ARIMA  13.75  17.48     23.31
2               Poliklinik Penyakit Dalam     LSTM  14.86  18.88     23.35
0               Poliklinik Penyakit Dalam  Prophet  16.15  19.61     30.16
4           Poliklinik Rehabilitasi Medik    ARIMA   4.25   5.33     13.61
5           Poliklinik Rehabilitasi Medik     LSTM   4.31   5.31     13.73
3       

In [ ]:
# ==========================================
# 5. RATA-RATA EVALUASI PER MODEL (BUKTI AKHIR)
# ==========================================

# Menghitung rata-rata metrik untuk setiap model dari seluruh poliklinik
df_average = df_final.groupby('Model')[['MAE', 'RMSE', 'MAPE (%)']].mean().reset_index()

# Mengurutkan berdasarkan MAE terkecil untuk menentukan model terbaik secara global
df_average = df_average.sort_values(by='MAE').reset_index(drop=True)

print("\n" + "="*70)
print("    TABEL RATA-RATA EVALUASI TOTAL (SELURUH POLIKLINIK)")
print("="*70)
print(df_average)
print("="*70)

# Menentukan model terbaik secara otomatis berdasarkan MAE rata-rata terkecil
best_model = df_average.iloc[0]['Model']
best_mae = df_average.iloc[0]['MAE']

print(f"\nKesimpulan: Model TERBAIK secara keseluruhan adalah **{best_model}**")
print(f"dengan rata-rata MAE sebesar {round(best_mae, 2)} pasien per hari.")

# Simpan tabel rata-rata ke Excel untuk data pendukung laporan
df_average.to_excel('Rata_Rata_Evaluasi_Model.xlsx', index=False)
print("✅ Tabel rata-rata berhasil disimpan sebagai 'Rata_Rata_Evaluasi_Model.xlsx'")


    TABEL RATA-RATA EVALUASI TOTAL (SELURUH POLIKLINIK)
     Model     MAE    RMSE  MAPE (%)
0    ARIMA   9.752  14.250    35.276
1     LSTM  11.040  14.990    47.418
2  Prophet  15.344  19.386    80.104

Kesimpulan: Model TERBAIK secara keseluruhan adalah **ARIMA**
dengan rata-rata MAE sebesar 9.75 pasien per hari.
✅ Tabel rata-rata berhasil disimpan sebagai 'Rata_Rata_Evaluasi_Model.xlsx'
